# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn --quiet

# 1. Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- LoRA ----
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    # ---- Training ----
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- BCE ----
    bce_weight: float = 1.0

    # ---- Entropy regularization ----
    # Rewards uncertainty: H(p) = -p*log(p) - (1-p)*log(1-p), maximized at p=0.5
    # Penalizes confident (collapsed) predictions, encouraging calibrated outputs
    lambda_entropy_reg: float = 0.1

    # ---- Prediction marker (plain text, NOT a special token) ----
    prediction_marker: str = "<|prediction|>"

    # ---- xVal: GPT prediction marker ----
    gpt_prediction_marker: str = "<|GPT_prediction|>"

    # ---- Paths ----
    output_dir: str = "./outputs"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

# 3. Load Training Data

In [ ]:
import pandas as pd
from datasets import Dataset

train_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/ft-0402-etl-with-gpt-pr/train.csv").iloc[:500]
train_data = train_data.rename(columns={"gpt_probability": "gpt_prob"})

assert not train_data["gpt_prob"].isna().any(), "gpt_prob has NaN values in train"

train_dataset = Dataset.from_pandas(train_data[["text", "gpt_prob"]])

print(f"Train: {len(train_dataset)} samples")

# 4. Load Model + Tokenizer (QLoRA)
No special tokens are added. The prediction marker `<|prediction|>` is treated as
regular text and tokenized into multiple sub-tokens by the base tokenizer.
This means we do NOT need `resize_token_embeddings` or `modules_to_save`,
which keeps the trainable parameter count small.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer (no special tokens added) ----
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- Encode the prediction marker as a multi-token sequence ----
prediction_marker_ids = tokenizer.encode(
    cfg.prediction_marker, add_special_tokens=False
)
print(f"Prediction marker '{cfg.prediction_marker}' -> token IDs: {prediction_marker_ids}")
print(f"  Decoded back: {[tokenizer.decode([t]) for t in prediction_marker_ids]}")

# ---- Encode the GPT prediction marker ----
gpt_prediction_marker_ids = tokenizer.encode(
    cfg.gpt_prediction_marker, add_special_tokens=False
)
print(f"GPT prediction marker '{cfg.gpt_prediction_marker}' -> token IDs: {gpt_prediction_marker_ids}")
print(f"  Decoded back: {[tokenizer.decode([t]) for t in gpt_prediction_marker_ids]}")

# ---- Get label token IDs for BCE loss ----
label_0_token_id = tokenizer.encode("0", add_special_tokens=False)[0]
label_1_token_id = tokenizer.encode("1", add_special_tokens=False)[0]

print(f"Token IDs -> '0': {label_0_token_id}, '1': {label_1_token_id}")
assert label_0_token_id != tokenizer.unk_token_id, "'0' mapped to UNK!"
assert label_1_token_id != tokenizer.unk_token_id, "'1' mapped to UNK!"

# ---- Load model (no embedding resize needed) ----
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16
)

print(f"Model loaded. Vocab size: {model.config.vocab_size}")

# 5. Apply LoRA
Without `modules_to_save`, only the LoRA adapter weights are trainable.
The full `embed_tokens` and `lm_head` layers stay frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.lora_target_modules,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# 5b. xVal Data Collator
Handles the scalar `gpt_prob` field that `DataCollatorForSeq2Seq` can't collate.
The xVal embedding scaling itself happens inside `BCETrainer.compute_loss`.

In [ ]:
import torch.nn as nn


class XValDataCollator:
    """Wraps base collator to handle gpt_prob and gpt_marker_start/end fields."""

    def __init__(self, base_collator):
        self.base_collator = base_collator

    def __call__(self, features):
        gpt_probs = [f.pop("gpt_prob") for f in features]
        gpt_marker_starts = [f.pop("gpt_marker_start") for f in features]
        gpt_marker_ends = [f.pop("gpt_marker_end") for f in features]
        batch = self.base_collator(features)
        batch["gpt_prob"] = torch.tensor(gpt_probs, dtype=torch.float32)
        batch["gpt_marker_start"] = torch.tensor(gpt_marker_starts, dtype=torch.long)
        batch["gpt_marker_end"] = torch.tensor(gpt_marker_ends, dtype=torch.long)
        return batch


print("XValDataCollator defined.")

# 6. Tokenize with Label Masking
Everything up to and including `<|prediction|>` is masked (`labels=-100`).
Only the label token (`0`/`1`) and closing marker are trained on.
This focuses all gradient signal on the classification task.

In [ ]:
from transformers import DataCollatorForSeq2Seq

mask_preserve = 50


def find_marker_token_range(text, marker_str, tokenizer, max_length):
    """Find the token index range [start, end) for a marker string in text.

    Uses character offsets from the tokenizer to map the marker's character
    span to token indices, avoiding token-boundary mismatch issues.
    Returns (start, end) or (-1, -1) if not found.
    """
    char_start = text.find(marker_str)
    if char_start == -1:
        return -1, -1
    char_end = char_start + len(marker_str)

    encoding = tokenizer(text, truncation=True, max_length=max_length, return_offsets_mapping=True)
    offsets = encoding["offset_mapping"]

    tok_start, tok_end = -1, -1
    for i, (cs, ce) in enumerate(offsets):
        if cs == ce == 0 and i > 0:
            continue  # skip special tokens
        if tok_start == -1 and ce > char_start:
            tok_start = i
        if cs < char_end:
            tok_end = i + 1

    return tok_start, tok_end


def tokenize_and_mask(examples):
    tokenized = tokenizer(
        examples["text"], truncation=True, max_length=cfg.max_seq_length
    )
    all_labels = []
    all_gpt_marker_start = []
    all_gpt_marker_end = []

    for i, input_ids in enumerate(tokenized["input_ids"]):
        text = examples["text"][i]
        labels = input_ids.copy()

        # Mask everything before prediction marker (minus mask_preserve)
        tok_start, tok_end = find_marker_token_range(
            text, cfg.prediction_marker, tokenizer, cfg.max_seq_length
        )
        if tok_start != -1:
            mask_end = max(0, tok_end - mask_preserve)
            labels[:mask_end] = [-100] * mask_end

        # Find GPT prediction marker range for xVal scaling
        gpt_start, gpt_end = find_marker_token_range(
            text, cfg.gpt_prediction_marker, tokenizer, cfg.max_seq_length
        )
        all_gpt_marker_start.append(gpt_start)
        all_gpt_marker_end.append(gpt_end)
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    tokenized["gpt_prob"] = examples["gpt_prob"]
    tokenized["gpt_marker_start"] = all_gpt_marker_start
    tokenized["gpt_marker_end"] = all_gpt_marker_end
    return tokenized


train_tokenized = train_dataset.map(tokenize_and_mask, batched=True, remove_columns=["text"])

data_collator = XValDataCollator(
    DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8)
)

# ---- Diagnostic: verify masking on first sample ----
sample = train_tokenized[0]
input_ids = sample["input_ids"]
labels = sample["labels"]
total = len(labels)
masked_count = sum(1 for l in labels if l == -100)
unmasked_ids = [t for t in labels if t != -100]

# Re-tokenize with offsets for diagnostic
sample_text = train_dataset[0]["text"]
pred_start, pred_end = find_marker_token_range(sample_text, cfg.prediction_marker, tokenizer, cfg.max_seq_length)
gpt_start = sample["gpt_marker_start"]
gpt_end = sample["gpt_marker_end"]

print(f"=== Masking Diagnostic (sample 0) ===")
print(f"Total tokens: {total}")
print(f"Masked (labels=-100): {masked_count}")
print(f"Unmasked (trained on): {total - masked_count}")
print(f"mask_preserve setting: {mask_preserve}")
print()
print(f"Prediction marker token range: [{pred_start}, {pred_end})")
if pred_start != -1:
    actual_mask_end = max(0, pred_end - mask_preserve)
    print(f"mask_end (pred_end - mask_preserve): {actual_mask_end}")
    print(f"Decoded marker tokens: '{tokenizer.decode(input_ids[pred_start:pred_end])}'")
print()
print(f"GPT prediction marker token range: [{gpt_start}, {gpt_end})")
if gpt_start != -1:
    gpt_masked = all(labels[i] == -100 for i in range(gpt_start, gpt_end))
    print(f"GPT marker tokens fully masked: {gpt_masked}")
    print(f"Decoded GPT marker tokens: '{tokenizer.decode(input_ids[gpt_start:gpt_end])}'")
print()
print(f"Unmasked text: '{tokenizer.decode(unmasked_ids)}'")
print(f"gpt_prob: {sample['gpt_prob']}")
print()

# Show the boundary region around mask_end
if pred_start != -1:
    start = max(0, actual_mask_end - 5)
    end = min(total, actual_mask_end + 10)
    print(f"--- Boundary region (tokens {start} to {end-1}) ---")
    for i in range(start, end):
        tok_text = tokenizer.decode([input_ids[i]])
        masked = "MASKED" if labels[i] == -100 else f"TRAIN (label={labels[i]})"
        marker = " <-- mask_end" if i == actual_mask_end else ""
        print(f"  [{i}] {tok_text!r:20s} {masked}{marker}")

# 7. BCE + Entropy Regularized Trainer
Uses standard `Trainer` (not SFTTrainer) since we pre-tokenized with custom labels.
LM loss only fires on unmasked positions (after `<|prediction|>`).
BCE loss targets the prediction marker position.

**Entropy regularization**: At the prediction marker position, we compute
$p = \sigma(\text{logit}_1 - \text{logit}_0)$ and add
$-\lambda \cdot H(p) = -\lambda \cdot [-p\log p - (1-p)\log(1-p)]$ to the loss.
Since we *minimize* loss, the negative sign means we *maximize* entropy,
penalizing overconfident predictions and preventing probability collapse.

In [ ]:
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss
from transformers import Trainer, TrainingArguments


def find_subseq(seq, subseq):
    """Find starting index of subseq in a list. Returns -1 if not found."""
    n = len(subseq)
    for i in range(len(seq) - n + 1):
        if seq[i:i + n] == subseq:
            return i
    return -1


def binary_entropy(p):
    """Compute H(p) = -p*log(p) - (1-p)*log(1-p), clamped for numerical stability."""
    p = p.clamp(1e-7, 1.0 - 1e-7)
    return -p * p.log() - (1.0 - p) * (1.0 - p).log()


class BCEEntropyTrainer(Trainer):
    """
    Extends Trainer with:
    1. xVal embedding scaling: multiplies <|GPT_prediction|> token embeddings by gpt_prob
    2. BCE loss on the <|prediction|> marker position
    3. Entropy regularization: -lambda * H(p) to prevent probability collapse
    """

    def __init__(self, *args, bce_weight=1.0, lambda_entropy_reg=0.1,
                 prediction_marker_ids=None,
                 label_0_token_id=None, label_1_token_id=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.bce_weight = bce_weight
        self.lambda_entropy_reg = lambda_entropy_reg
        self.prediction_marker_ids = prediction_marker_ids
        self.label_0_token_id = label_0_token_id
        self.label_1_token_id = label_1_token_id
        self.bce_loss_fn = BCEWithLogitsLoss()

    def _xval_forward(self, model, input_ids, gpt_prob, gpt_marker_start, gpt_marker_end, **kwargs):
        """Convert input_ids to embeddings, scale GPT marker positions, forward."""
        embed_layer = model.get_input_embeddings()
        inputs_embeds = embed_layer(input_ids)

        batch_size, seq_len = input_ids.shape
        scale = torch.ones(batch_size, seq_len,
                           dtype=inputs_embeds.dtype, device=inputs_embeds.device)

        gpt_prob = gpt_prob.to(dtype=inputs_embeds.dtype, device=inputs_embeds.device)
        for b in range(batch_size):
            s, e = gpt_marker_start[b].item(), gpt_marker_end[b].item()
            if s != -1:
                scale[b, s:e] = gpt_prob[b]

        inputs_embeds = inputs_embeds * scale.unsqueeze(-1)
        return model(inputs_embeds=inputs_embeds, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        input_ids = inputs.pop("input_ids")
        gpt_prob = inputs.pop("gpt_prob")
        gpt_marker_start = inputs.pop("gpt_marker_start")
        gpt_marker_end = inputs.pop("gpt_marker_end")

        # ---- xVal forward: scale GPT marker embeddings by gpt_prob ----
        outputs = self._xval_forward(
            model, input_ids, gpt_prob, gpt_marker_start, gpt_marker_end,
            attention_mask=inputs.get("attention_mask"),
        )
        logits = outputs.logits

        # ---- Masked LM loss (only on unmasked positions) ----
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        lm_loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )

        # ---- BCE loss + entropy regularization on prediction marker position ----
        batch_size = input_ids.size(0)
        marker_ids = self.prediction_marker_ids

        bce_losses = []
        entropies = []
        for b in range(batch_size):
            seq = input_ids[b].tolist()
            start_idx = find_subseq(seq, marker_ids)

            if start_idx == -1:
                continue

            pred_pos = start_idx + len(marker_ids) - 1

            if pred_pos + 1 < input_ids.size(1):
                logit_0 = logits[b, pred_pos, self.label_0_token_id]
                logit_1 = logits[b, pred_pos, self.label_1_token_id]

                actual_next = input_ids[b, pred_pos + 1].item()
                target = torch.tensor(
                    1.0 if actual_next == self.label_1_token_id else 0.0,
                    device=logits.device,
                )

                # BCE loss
                bce_logit = logit_1 - logit_0
                bce_losses.append(self.bce_loss_fn(bce_logit, target))

                # Entropy: H(p) where p = sigmoid(logit_1 - logit_0)
                p = torch.sigmoid(bce_logit)
                entropies.append(binary_entropy(p))

        if bce_losses:
            bce_loss_mean = torch.stack(bce_losses).mean()
            entropy_mean = torch.stack(entropies).mean()
            # Subtract entropy to maximize it (we minimize loss)
            total_loss = lm_loss + self.bce_weight * bce_loss_mean - self.lambda_entropy_reg * entropy_mean
        else:
            total_loss = lm_loss
            bce_loss_mean = torch.tensor(0.0, device=lm_loss.device)
            entropy_mean = torch.tensor(0.0, device=lm_loss.device)

        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "lm_loss": lm_loss.detach().item(),
                "bce_loss": bce_loss_mean.detach().item(),
                "entropy": entropy_mean.detach().item(),
                "total_loss": total_loss.detach().item(),
            })

        return (total_loss, outputs) if return_outputs else total_loss

# 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    remove_unused_columns=False,
    seed=42,
    report_to="none",
)

trainer = BCEEntropyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
    bce_weight=cfg.bce_weight,
    lambda_entropy_reg=cfg.lambda_entropy_reg,
    prediction_marker_ids=prediction_marker_ids,
    label_0_token_id=label_0_token_id,
    label_1_token_id=label_1_token_id,
)

print(f"Entropy regularization lambda: {cfg.lambda_entropy_reg}")
print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

# 9. Save LoRA Adapter

In [ ]:
import os, json

adapter_path = f"{cfg.output_dir}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

with open(os.path.join(adapter_path, "experiment_config.json"), "w") as fp:
    json.dump(vars(cfg), fp, indent=2)